# HRSBF — A100 Worker integrado\n\nExecute a célula abaixo uma vez. Ela sincroniza `colab-a100-2026-10-06`, confirma a GPU e inicia o worker.\n\nO Secret `GITHUB_TOKEN` precisa estar habilitado para este notebook.

In [ ]:
from google.colab import userdata\nimport os, pathlib, subprocess, sys\nREPO='https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git'\nBRANCH='colab-a100-2026-10-06'\nROOT=pathlib.Path('/content/hrsbf-bent-verification')\ntoken=userdata.get('GITHUB_TOKEN')\nif not token: raise RuntimeError('GITHUB_TOKEN não encontrado no Secrets ou sem acesso ao notebook.')\nprint('1/5 GitHub')\nsubprocess.run(['gh','auth','login','--hostname','github.com','--with-token'],input=token,text=True,check=True)\nsubprocess.run(['gh','auth','setup-git'],check=True)\nprint('2/5 Repo')\nif not (ROOT/'.git').exists(): subprocess.run(['git','clone',REPO,str(ROOT)],check=True)\nsubprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)\nsubprocess.run(['git','-C',str(ROOT),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)\nos.chdir(ROOT)\nprint('3/5 Branch',subprocess.check_output(['git','branch','--show-current'],text=True).strip())\nprint('SHA',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())\nprint('4/5 GPU')\ngpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True,capture_output=True)\nprint(gpu.stdout.strip() if gpu.returncode==0 else 'GPU NÃO DETECTADA nesta sessão')\nprint('5/5 Worker — deixe esta célula rodando')\nenv={**os.environ,'COLAB_BRANCH':BRANCH}\nsubprocess.run([sys.executable,'colab_worker.py','--poll','30'],check=True,env=env)\n